In [6]:
import polars as pl

In [7]:
tie_lines = pl.read_excel(r"C:\Juan\Maestría\Code\oleosim\ExperimentalData\data\No 47exp\refined_lle_data_v5.xlsx", engine="calamine")
comps_MW = pl.read_excel(r"C:\Juan\Maestría\Code\oleosim\ExperimentalData\data\comps_MW.xlsx", engine="calamine")

In [8]:
comps_MW =comps_MW.drop(["Type Alcohol", "MW_Glycerol", "MW_Alcohol", "MW_H2O"])

In [9]:
complete_data = tie_lines.join(comps_MW, on=["Ref", "Sub Ref"], how="left")

In [10]:
complete_data = complete_data.drop(["Ref", "Sub Ref", "Reconcile phi"])

In [11]:
complete_data

Temperature (K),Type Alcohol,FAME/FAEE_F,GLYCEROL_F,ALCOHOL_F,OIL_F,DAG_F,MAG_F,ACID_F,WATER_F,FAME/FAEE_L,GLYCEROL_L,ALCOHOL_L,OIL_L,DAG_L,MAG_L,ACID_L,WATER_L,FAME/FAEE_H,GLYCEROL_H,ALCOHOL_H,OIL_H,DAG_H,MAG_H,ACID_H,WATER_H,MW_FAME/FAEE,MW_Oil,MW_DAG,MW_MAG,MW_FA
f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
298.15,1,0.4,0.4,0.2,0.0,0.0,0.0,0.0,0.0,0.881461,0.010995,0.106827,0.0,0.0,0.0,0.0,0.0,0.00001,0.723179,0.277407,0.0,0.0,0.0,0.0,0.0,324.139909,968.419726,659.279817,350.139909,296.139909
298.15,1,0.385,0.385,0.23,0.0,0.0,0.0,0.0,0.0,0.856402,0.028158,0.115568,0.0,0.0,0.0,0.0,0.0,0.002266,0.674722,0.322908,0.0,0.0,0.0,0.0,0.0,324.139909,968.419726,659.279817,350.139909,296.139909
298.15,1,0.41,0.41,0.18,0.0,0.0,0.0,0.0,0.0,0.90767,0.005553,0.086762,0.0,0.0,0.0,0.0,0.0,0.000146,0.74308,0.256786,0.0,0.0,0.0,0.0,0.0,324.139909,968.419726,659.279817,350.139909,296.139909
293.15,2,0.4431,0.438,0.1189,0.0,0.0,0.0,0.0,0.0,0.895378,0.024504,0.080139,0.0,0.0,0.0,0.0,0.0,0.026639,0.818751,0.154592,0.0,0.0,0.0,0.0,0.0,315.953619,901.860858,614.907238,327.953619,273.953619
293.15,2,0.4176,0.4162,0.1662,0.0,0.0,0.0,0.0,0.0,0.873731,0.023739,0.102309,0.0,0.0,0.0,0.0,0.0,0.025474,0.75359,0.221126,0.0,0.0,0.0,0.0,0.0,315.953619,901.860858,614.907238,327.953619,273.953619
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
318.2,2,0.0,0.0,0.447,0.513,0.0,0.0,0.0,0.041,0.0,0.0,0.109755,0.888876,0.0,0.0,0.0,0.002904,0.0,0.0,0.866894,0.045007,0.0,0.0,0.0,0.088432,245.611894,690.835683,474.223789,257.611894,203.611894
318.2,2,0.0,0.0,0.494,0.496,0.0,0.0,0.0,0.01,0.0,0.0,0.342187,0.651845,0.0,0.0,0.0,0.005811,0.0,0.0,0.718954,0.265071,0.0,0.0,0.0,0.016207,245.611894,690.835683,474.223789,257.611894,203.611894
318.2,2,0.0,0.0,0.501,0.479,0.0,0.0,0.0,0.02,0.0,0.0,0.239539,0.754497,0.0,0.0,0.0,0.006396,0.0,0.0,0.818388,0.144574,0.0,0.0,0.0,0.036514,245.611894,690.835683,474.223789,257.611894,203.611894


In [14]:
column_pairs = [
    ("FAME/FAEE_F", "MW_FAME/FAEE"),
    ("OIL_F", "MW_Oil"),
    ("DAG_F", "MW_DAG"),
    ("MAG_F", "MW_MAG"),
    ("ACID_F", "MW_FA")
]

# For each pair, set MW to 0 if the concentration is 0
for conc_col, mw_col in column_pairs:
    complete_data = complete_data.with_columns([
        pl.when(pl.col(conc_col) == 0)
        .then(0)
        .otherwise(pl.col(mw_col))
        .alias(mw_col)
    ])

In [17]:
complete_data = complete_data.fill_null(0)

In [19]:
complete_data.write_csv("exp_data.csv")